# Modelo de Churn / Attrition Bancario — Pipeline CompletoPanel temporal CIF×mes → feature engineering → modelos → ensemble → explicabilidad.**Cómo usarlo:**1. Corre las celdas de *Setup* (instala dependencias y monta Drive).2. En `config.yaml`, `data.source` controla todo:   - `synthetic` → genera datos simulados. **Empieza aquí**: valida el pipeline sin datos reales.   - `drive` → lee los extractos del agente SQL desde Drive.3. El resto corre igual en ambos casos.**Dónde van los datos reales en Drive:**```MyDrive/churn_model/data/raw/    q1_party_master.csv        q5_txn_monthly.csv    q2_cif_account_relationship.csv   q6_digital_contact_monthly.csv    q3_deposit_monthly.csv     q7_product_events.csv    q4_loan_monthly.csv```---## Setup

In [ ]:
# Colab ya trae numpy/pandas/sklearn/torch/xgboost!pip install -q lightgbm catboost shap pyyaml 2>/dev/nullprint("✓ dependencias listas")

In [ ]:
# ── Montar Google Drive ──# Fuera de Colab esto no hace nada, así que la misma celda sirve en local.import sysIN_COLAB = "google.colab" in sys.modulesif IN_COLAB:    from google.colab import drive    drive.mount("/content/drive")    print("✓ Drive montado")else:    print("ℹ️  Fuera de Colab — se omite el montaje")

In [ ]:
# ── Traer el código ──# En Colab clona el repo. En local usa la carpeta actual.# Si prefieres tener el código en Drive, apunta PROJECT_DIR ahí y omite el clone.import os, subprocess, sysfrom pathlib import PathREPO_URL = "https://github.com/DanielRegaladoUMiami/bank-churn-attrition-design.git"if IN_COLAB:    PROJECT_DIR = Path("/content/churn")    if not PROJECT_DIR.exists():        subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)else:    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()os.chdir(PROJECT_DIR)sys.path.insert(0, str(PROJECT_DIR / "src"))print(f"✓ Proyecto: {PROJECT_DIR}")

In [ ]:
import warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltpd.set_option("display.max_columns", 60)pd.set_option("display.width", 200)plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True,                     "grid.alpha": 0.25, "font.size": 10})from churn.config import Configfrom churn.data import load_tables, profile, validatefrom churn.panel import build_cif_month_state, label_attrition_state, build_panelfrom churn.features import build_features, select_feature_columns, clean_featuresfrom churn.evaluation import (temporal_split, evaluate, summarize, decile_table,                              calibration_table, calibrate, make_cv)from churn.models import tune_all, build_stack, build_sequences, SequenceModel, SEQUENCE_COLSfrom churn.explain import (shap_analysis, global_importance, explain_customer,                           counterfactual_analysis, sanity_check)cfg = Config.load("config.yaml")# ⬇️ CAMBIA ESTO cuando tengas los extractos reales en Drivecfg.data["source"] = "synthetic"      # "synthetic" | "drive"print(f"Fuente de datos: {cfg.data['source']}")print(f"Ventanas: feature={cfg.panel['feature_window_months']}m  "      f"gap={cfg.panel['gap_months']}m  "      f"performance={cfg.panel['performance_window_months']}m")

---## 1 · Carga y validación`load_tables` devuelve las 7 tablas con el mismo esquema vengan de donde vengan.`validate` las contrasta contra el contrato de `churn/config.py` — llaves duplicadas,columnas obligatorias faltantes, nulos por encima del umbral.

In [ ]:
tables = load_tables(cfg)profile(tables)

In [ ]:
_ = validate(tables)

---## 2 · EDALo que importa mirar antes de modelar:1. **Aperturas y cierres por mes** — un pico aislado casi siempre es una conversión de   core, no comportamiento de clientes. Si aparece, hay que cortar la historia ahí.2. **Distribución de saldos** — define el umbral de "saldo bajo" del target. No lo   inventes: léelo de aquí.3. **Actividad transaccional** — define el umbral de inactividad.4. **Mix de productos y roles** — cuánto pesan las cuentas joint y los authorized signers.

In [ ]:
# ── 1. Aperturas vs cierres por mes: detector de conversión de core ──ev = tables["events"]opens = (ev[ev.event_type == "account_open"].groupby(ev.event_date.dt.to_period("M"))         .size().rename("aperturas"))closes = (ev[ev.event_type == "account_close"].groupby(ev.event_date.dt.to_period("M"))          .size().rename("cierres"))flow = pd.concat([opens, closes], axis=1).fillna(0)flow = flow[flow.index >= tables["deposits"].as_of_month.min().to_period("M")]fig, ax = plt.subplots(figsize=(12, 4))flow.plot(ax=ax, marker="o", ms=3)ax.set_title("Aperturas y cierres por mes — un pico aislado = posible conversión de core")ax.set_xlabel(""); ax.legend()plt.tight_layout(); plt.show()z = (flow.cierres - flow.cierres.mean()) / flow.cierres.std()if (z.abs() > 3).any():    print(f"⚠️  Meses con cierres anómalos (|z|>3): {list(flow.index[z.abs() > 3])}")else:    print("✓ Sin picos anómalos de cierres")

In [ ]:
# ── 2. Distribución de saldos: de aquí sale el umbral del target ──dep = tables["deposits"]bal = dep.loc[dep.eom_balance > 0, "eom_balance"]fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))axes[0].hist(np.log10(bal.clip(lower=1)), bins=60, color="steelblue")axes[0].set_title("log10(saldo) — cuentas con saldo > 0"); axes[0].set_xlabel("log10 USD")for p in (0.5, 1, 2, 3):    axes[0].axvline(p, color="crimson", ls="--", lw=0.8)qs = bal.quantile([.05, .10, .25, .50, .75, .90, .99])axes[1].axis("off")axes[1].table(cellText=[[f"p{int(i*100)}", f"${v:,.0f}"] for i, v in qs.items()],              colLabels=["percentil", "saldo"], loc="center", cellLoc="left")axes[1].set_title("Percentiles de saldo")plt.tight_layout(); plt.show()print(f"Cuentas con saldo < $100:  {(dep.eom_balance < 100).mean():.1%}")print(f"Cuentas con saldo == 0:    {(dep.eom_balance == 0).mean():.1%}")print(f"\n→ El umbral `target.low_balance_abs` (hoy "      f"${cfg.target['low_balance_abs']:,.0f}) debe caer en la cola izquierda,")print("  no cortar clientes normales.")

In [ ]:
# ── 3. Mix de productos y de roles del CIF ──rel = tables["relationships"]fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))dep.product_family.value_counts().plot.barh(ax=axes[0], color="steelblue")axes[0].set_title("Cuentas-mes por familia de producto")rc = rel.relationship_role.value_counts()rc.plot.barh(ax=axes[1], color="darkorange")axes[1].set_title("Relaciones por rol del CIF")plt.tight_layout(); plt.show()share = (rc / rc.sum()).mul(100).round(1)print("Peso de cada rol (%):")print(share.to_string())print(f"\n→ {share.get('joint', 0):.1f}% joint: si sumaras saldos sin ponderar por")print("  ownership, inflarías los depósitos del banco en esa proporción.")

---## 3 · Panel y targetAquí se materializan las decisiones de diseño:- Saldos **ponderados por rol** (joint = 1/n titulares, authorized signer = 0).- Target **compuesto**: cierre formal **o** (saldo bajo + sin préstamos + inactividad),  sostenido N meses — para capturar la attrition silenciosa.- **Espina = maestro de clientes**, no relaciones vigentes: si no, el cliente que cierra  todo desaparece del panel en vez de aparecer con ceros.

In [ ]:
state = build_cif_month_state(tables, cfg)state_lab = label_attrition_state(state, cfg)print(f"Estado CIF×mes: {state.shape[0]:,} filas × {state.shape[1]} columnas")n = state_lab.cif_id.nunique()ev_n = state_lab.dropna(subset=["churn_event_month"]).cif_id.nunique()print(f"CIFs con evento de churn detectado: {ev_n:,} / {n:,} ({ev_n/n:.1%})")

In [ ]:
# Descomposición del target: ¿qué condición está mandando?conds = {    "saldo bajo": state_lab.owned_balance < np.maximum(        cfg.target["low_balance_abs"],        state_lab.balance_baseline_12m.fillna(0) * cfg.target["low_balance_pct_of_baseline"]),    "sin préstamos activos": state_lab.n_active_loans.fillna(0) <= 0,    "inactivo ≥3m": state_lab.months_since_activity.fillna(0) >= 3,    "sin relaciones vigentes": state_lab.n_relationships_any_role.fillna(0) <= 0,    "ESTADO attrited (combinado)": state_lab.is_attrited_state,}pd.Series({k: f"{v.mean():.2%}" for k, v in conds.items()}, name="% de filas del panel").to_frame()

In [ ]:
panel = build_panel(tables, cfg)# Estabilidad de la tasa base en el tiempo: si salta, hay un problema de datosrate = panel.groupby("as_of_month").y.agg(["mean", "size"])fig, ax = plt.subplots(figsize=(12, 3.5))ax.plot(rate.index, rate["mean"] * 100, marker="o", ms=4, color="crimson")ax.axhline(panel.y.mean() * 100, ls="--", color="gray", lw=1,           label=f"promedio {panel.y.mean():.2%}")ax.set_title("Tasa de churn por mes de observación"); ax.set_ylabel("% churn")ax.legend(); plt.tight_layout(); plt.show()

---## 4 · Feature engineeringPara cada fila `(cif, as_of_month)`, **todo** se calcula con la ventana hacia atrás`[t-11, t]`. Nunca del futuro. Nueve familias: nivel, tendencia (pendiente OLS),ratio a sí mismo, **drawdown**, volatilidad, **aceleración**, rachas, estacionalidady relacionales (contagio de co-titular).

In [ ]:
feats = build_features(panel, cfg, tables)numeric, categorical = select_feature_columns(feats)feats = clean_features(feats, numeric)print(f"{len(numeric)} numéricas + {len(categorical)} categóricas")print("\nMuestra por familia:")for fam, pat in [("nivel", "owned_balance$"), ("tendencia", "slope"),                 ("drawdown", "drawdown"), ("aceleración", "accel"),                 ("racha", "streak"), ("dominio", "dd_lost|loan_ending|cd_maturing"),                 ("relacional", "co_owner|n_co")]:    cols = [c for c in numeric if pd.Series([c]).str.contains(pat, regex=True)[0]]    print(f"  {fam:14s} {len(cols):3d}  ej: {cols[:3]}")

---## 5 · Split out-of-time con embargoDos protecciones contra leakage:- **Corte temporal**, no aleatorio: los últimos N meses son el holdout.- **Embargo**: se eliminan del train las filas cuya ventana de etiqueta invade el  periodo de test. Cuesta `gap + performance` meses de panel, pero sin él el holdout  está contaminado.

In [ ]:
train, test = temporal_split(feats, cfg)Xtr, ytr = train[numeric + categorical], train.y.to_numpy()Xte, yte = test[numeric + categorical], test.y.to_numpy()groups = train.cif_id.to_numpy()   # ← clave: el CV agrupa por esto

---## 6 · Tuning con `StratifiedGroupKFold``groups=cif_id` es lo que impide que el mismo cliente esté en train y validación.Con un `StratifiedKFold` normal el PR-AUC sube varios puntos sin que el modelo mejore.La métrica de optimización es **average precision (PR-AUC)**, no accuracy ni ROC-AUC.

In [ ]:
%%timesearches = tune_all(Xtr, ytr, groups, numeric, categorical, cfg)results, fitted = {}, {}for name, s in searches.items():    p = s.best_estimator_.predict_proba(Xte)[:, 1]    results[name] = evaluate(yte, p)    fitted[name] = s.best_estimator_summarize(results)

---## 7 · Stacking

In [ ]:
stack, chosen = build_stack(searches, cfg.models["stack_top_k"], cfg)if stack is not None:    print(f"Base learners: {chosen}")    stack.fit(Xtr, ytr)    results["stacking"] = evaluate(yte, stack.predict_proba(Xte)[:, 1])    fitted["stacking"] = stacksummarize(results)

---## 8 · Red secuencial (GRU)El GBM ve *resúmenes* de la trayectoria (pendientes, drawdowns). El GRU ve la**serie mensual cruda** `[muestras, meses, canales]` y aprende la forma completa.El ensemble que aporta es la mezcla de ambos, no cinco GBMs promediados.

In [ ]:
%%timewin = cfg.panel["feature_window_months"]Str = build_sequences(state, train, SEQUENCE_COLS, win)Ste = build_sequences(state, test, SEQUENCE_COLS, win)print(f"Tensor: {Str.shape}  [muestras, meses, canales]")sq = cfg.models["sequence_model"]gru = SequenceModel(n_features=Str.shape[2], hidden_size=sq["hidden_size"],                    num_layers=sq["num_layers"], dropout=sq["dropout"],                    lr=sq["lr"], epochs=sq["epochs"], batch_size=sq["batch_size"],                    patience=sq["patience"]).fit(Str, ytr, Ste, yte)p_gru = gru.predict_proba(Ste)[:, 1]results["gru_sequence"] = evaluate(yte, p_gru)# Blend con el mejor tabularbest_tab = max([k for k in fitted], key=lambda k: results[k]["pr_auc"])p_tab = fitted[best_tab].predict_proba(Xte)[:, 1]for w in (0.2, 0.3, 0.4):    results[f"blend_{best_tab}+gru_{w:.0%}"] = evaluate(yte, (1 - w) * p_tab + w * p_gru)summarize(results)

---## 9 · Resultados**Cómo leer esto:** ignora accuracy. Con una tasa base de 2-5%, lo que importa es`lift@10` (cuántas veces mejor que aleatorio en el top 10%) y `capture@10` (quéfracción de todos los churners cae en ese top 10%). Retención solo puede llamar aesa lista.

In [ ]:
board = summarize(results)best_name = board.index[0]best_model = fitted.get(best_name, fitted[max(fitted, key=lambda k: results[k]["pr_auc"])])p_best = best_model.predict_proba(Xte)[:, 1]print(f"🏆 {best_name}: PR-AUC {board.loc[best_name,'pr_auc']:.4f}, "      f"lift@10 {board.loc[best_name,'lift@10']:.1f}x")board

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curvedt = decile_table(yte, p_best)fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))axes[0].bar(dt.decil, dt.lift, color="steelblue")axes[0].axhline(1, color="crimson", ls="--", lw=1)axes[0].set_title("Lift por decil"); axes[0].set_xlabel("decil (1 = mayor riesgo)")axes[1].plot(dt.decil * 10, dt["capture_%"], marker="o", color="darkorange")axes[1].plot([0, 100], [0, 100], ls="--", color="gray", lw=1, label="aleatorio")axes[1].set_title("Curva de captura"); axes[1].set_xlabel("% de la base contactada")axes[1].set_ylabel("% de churners capturados"); axes[1].legend()pr, rc, _ = precision_recall_curve(yte, p_best)axes[2].plot(rc, pr, color="seagreen")axes[2].axhline(yte.mean(), ls="--", color="gray", lw=1, label=f"base {yte.mean():.2%}")axes[2].set_title("Curva Precision-Recall"); axes[2].set_xlabel("recall")axes[2].set_ylabel("precision"); axes[2].legend()plt.tight_layout(); plt.show()dt

---## 10 · CalibraciónNecesaria si la probabilidad va a alimentar un cálculo de valor en riesgo(`P(churn) × saldo × margen`). El stacking suele descalibrar.

In [ ]:
n_cal = len(Xte) // 3cal = calibrate(best_model, Xte.iloc[:n_cal], yte[:n_cal])p_cal = cal.predict_proba(Xte.iloc[n_cal:])[:, 1]y_ev = yte[n_cal:]before, after = calibration_table(y_ev, p_best[n_cal:]), calibration_table(y_ev, p_cal)fig, ax = plt.subplots(figsize=(5, 4.5))ax.plot(before.prob_predicha, before.tasa_observada, "o-", label="sin calibrar")ax.plot(after.prob_predicha, after.tasa_observada, "s-", label="isotónica")lim = max(before.prob_predicha.max(), after.prob_predicha.max()) * 1.1ax.plot([0, lim], [0, lim], "--", color="gray", lw=1, label="perfecta")ax.set_xlabel("probabilidad predicha"); ax.set_ylabel("tasa observada")ax.set_title("Curva de calibración"); ax.legend()plt.tight_layout(); plt.show()print(f"Brier: {evaluate(y_ev, p_best[n_cal:])['brier']:.5f} → "      f"{evaluate(y_ev, p_cal)['brier']:.5f}")

---## 11 · Explicabilidad (SHAP)`TreeExplainer` es exacto y determinista sobre GBMs. **No usamos LIME**: reajusta unmodelo lineal local con muestreo aleatorio, así que dos corridas sobre el mismo clientedan explicaciones distintas — inaceptable si la explicación puede terminar en unexpediente de model risk (SR 11-7).El `sanity_check` avisa si aparece una variable geográfica o un identificador en eltop-10: casi siempre es un proxy de algo, o leakage.

In [ ]:
sv, Xt = shap_analysis(best_model, Xte, cfg)imp = global_importance(sv, Xt, top_n=25)fig, ax = plt.subplots(figsize=(9, 7))top = imp.head(20).iloc[::-1]colors = ["crimson" if d > 0.05 else "steelblue" if d < -0.05 else "gray"          for d in top.direccion]ax.barh(top.feature, top.shap_abs_mean, color=colors)ax.set_xlabel("|SHAP| medio"); ax.set_title("Importancia global (rojo = sube riesgo)")plt.tight_layout(); plt.show()_ = sanity_check(imp, expected_top=["direct_deposit", "balance", "drawdown", "slope"])imp

In [ ]:
# Explicación local: un cliente concreto del top de riesgotop_idx = Xte.index[np.argsort(p_best)[::-1][:1]][0]if top_idx in Xt.index:    display(explain_customer(best_model, Xte, top_idx, top_n=12,                             shap_values=sv, Xt=Xt))else:    print("El cliente de mayor riesgo no cayó en la muestra SHAP; "          "sube `explain.shap_sample_size` para incluirlo.")

---## 12 · Contrafactuales accionablesSHAP dice *qué pesó*. Retención necesita *qué hacer*. Esto simula la intervenciónsobre los clientes de mayor riesgo y mide la caída de probabilidad.

In [ ]:
cf = counterfactual_analysis(best_model, Xte, cfg)cf

---## 13 · Lista de scoring y guardadoLo que se entrega al equipo de retención: el top de riesgo con su probabilidadcalibrada, el saldo en riesgo y los drivers principales.

In [ ]:
import joblibscored = test[["cif_id", "as_of_month", "owned_balance", "y"]].copy()scored["p_churn"] = p_bestscored["valor_en_riesgo"] = scored.p_churn * scored.owned_balancescored = scored.sort_values("p_churn", ascending=False)out = cfg.artifacts_path()scored.head(500).to_csv(out / "lista_retencion_top500.csv", index=False)board.to_csv(out / "leaderboard.csv")imp.to_csv(out / "shap_importance.csv", index=False)joblib.dump(best_model, out / "modelo_final.joblib")print(f"💾 Guardado en {out}")print(f"\nValor total en riesgo (top 10%): "      f"${scored.head(len(scored)//10).valor_en_riesgo.sum():,.0f}")scored.head(15)

---## Siguiente paso1. Cambia `cfg.data["source"] = "drive"` y sube los extractos del agente SQL.2. Revisa la celda de EDA de **saldos** y ajusta `target.low_balance_abs` en `config.yaml`   a lo que digan tus datos.3. Revisa el gráfico de **aperturas/cierres**: si hay un pico, corta la historia ahí.4. Corre la descomposición del target — si una sola condición domina, la definición   está desbalanceada.5. Antes de presentar: valida el SHAP contra intuición de negocio y corre un análisis   de proxies sobre `state` y `preferred_language` (Fair Lending / UDAAP).